# 📅 Interactive Date Range Calendar
Hover over any event pill to see full details.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import date
import calendar
import json

## ⚙️ Configuration — edit only this cell

In [ ]:
# ── DATE RANGE ──────────────────────────────────────────────
START_DATE = '2026-07-09'
END_DATE   = '2026-08-02'

# ── EVENTS  {date: [(label, time, category, description)]} ──
# time: 'HH:MM' or None for all-day

EVENTS = {
    '2026-07-09': [('Llegada a GT',          '21:00', 'personal', 'Llegada a aeropuerto de Guatemala')],
    '2026-07-11': [('Antigua',               None,    'events', 'Salida con COD ida a la Antigua (día completo)')],
    '2026-07-12': [('Antigua',               None,    'events', 'Salida con COD regreso de la Antigua (día completo)')],
    '2026-07-13': [('Ale',             None,    'personal', 'Pasar el dia con Ale en sus vacaciones (día completo)')],
    '2026-07-14': [('Ale',             None,    'personal', 'Pasar el dia con Ale en sus vacaciones (día completo)')],
    '2026-07-15': [('Ale',             None,    'personal', 'Pasar el dia con Ale en sus vacaciones (día completo)')],
    '2026-07-16': [('Ale',             None,    'personal', 'Pasar el dia con Ale en sus vacaciones (día completo)')],
    '2026-07-17': [('Puerto',                None,    'events', 'Salida con Familia a Puerto (día completo)')],
    '2026-07-18': [('Puerto',                None,    'events', 'Dia completo en Puerto con Familia')],
    '2026-07-19': [('Puerto',                None,    'events', 'Regreso con Familia del Puerto (día completo)')],
    '2026-07-21': [('Dentista',              '10:00', 'doctors',  'Cita con dentista para limpieza, revision y blanqueamiento')],
    '2026-07-22': [
        ('Oftalmólogo',          '10:00', 'doctors',  'Cita con Julio Paz para revision de ojos'),
        ('Graduacion',           '16:00', 'events',   'Graduacion de Michelle en URL'),
    ],
    '2026-07-25': [('Fiesta de graduacion',  None,    'events',   'Fiesta de graduacion de Michelle')],
    '2026-08-02': [('Salida de GT',          '22:50', 'personal', 'Salida de aeropuerto de Guatemala')],
}

# ── COLORS ──────────────────────────────────────────────────
CATEGORY_COLORS = {
    'work':     '#378ADD',
    'personal': '#1D9E75',
    'doctors': '#D85A30',
    'events':  '#7F77DD',
}

MONTHS_PER_ROW = 2

# ── PILL LAYOUT — tweak if events overlap ───────────────────
PILL_HEIGHT    = 0.18   # height of each event pill
PILL_GAP       = 0.06   # gap between pills
MAX_PILLS      = 3      # max pills shown per cell (rest shown as +N)
DAY_NUM_H      = 0.22   # space reserved at top for the day number
CELL_PADDING   = 0.08   # padding inside cell top/bottom

## 🔧 Builder

In [3]:
def get_months_in_range(start, end):
    months, y, m = [], start.year, start.month
    while (y, m) <= (end.year, end.month):
        months.append((y, m))
        m += 1
        if m > 12: m, y = 1, y + 1
    return months

def axis_ref(idx):
    return '' if idx == 1 else str(idx)

def compute_cell_height():
    """Dynamic cell height based on pill layout settings."""
    return DAY_NUM_H + MAX_PILLS * (PILL_HEIGHT + PILL_GAP) + CELL_PADDING * 2

def build_month(year, month, start, end, events, today, axis_idx, months_per_row):
    cal_weeks  = calendar.monthcalendar(year, month)
    n_weeks    = len(cal_weeks)
    month_name = calendar.month_name[month]
    HEADER_H   = 0.70
    DAYLAB_H   = 0.45
    CELL_H     = compute_cell_height()
    TOTAL_H    = HEADER_H + DAYLAB_H + n_weeks * CELL_H
    suf        = axis_ref(axis_idx)
    xr, yr     = f'x{suf}', f'y{suf}'

    shapes, annotations, traces_with_axes = [], [], []
    row_num = (axis_idx - 1) // months_per_row + 1
    col_num = (axis_idx - 1) %  months_per_row + 1

    # ── month header ──────────────────────────────────────────
    shapes.append(dict(
        type='rect', xref=xr, yref=yr,
        x0=0, x1=7, y0=TOTAL_H - HEADER_H, y1=TOTAL_H,
        fillcolor='#2C2C2A', line_width=0, layer='below'
    ))
    annotations.append(dict(
        xref=xr, yref=yr, x=3.5, y=TOTAL_H - HEADER_H / 2,
        text=f'<b>{month_name} {year}</b>',
        font=dict(color='white', size=14), showarrow=False
    ))

    # ── weekday labels ────────────────────────────────────────
    for i, d in enumerate(['Mon','Tue','Wed','Thu','Fri','Sat','Sun']):
        color = '#A32D2D' if i >= 5 else '#555553'
        annotations.append(dict(
            xref=xr, yref=yr,
            x=i + 0.5, y=TOTAL_H - HEADER_H - DAYLAB_H / 2,
            text=f'<b>{d}</b>', font=dict(color=color, size=10), showarrow=False
        ))

    # ── day cells ─────────────────────────────────────────────
    for week_idx, week in enumerate(cal_weeks):
        yb = TOTAL_H - HEADER_H - DAYLAB_H - (week_idx + 1) * CELL_H
        yt = yb + CELL_H

        for dow, day in enumerate(week):
            x0, x1 = dow, dow + 1

            if day == 0:
                shapes.append(dict(
                    type='rect', xref=xr, yref=yr,
                    x0=x0+0.03, x1=x1-0.03, y0=yb+0.03, y1=yt-0.03,
                    fillcolor='#EFEFEF', line=dict(color='#D3D1C7', width=0.5),
                    layer='below'
                ))
                continue

            cell_date  = date(year, month, day)
            in_range   = start <= cell_date <= end
            is_today   = cell_date == today
            is_weekend = dow >= 5
            ds         = cell_date.strftime('%Y-%m-%d')
            day_evts   = events.get(ds, [])

            bg  = '#F8F8F6' if in_range else '#EFEFEF'
            ec  = '#378ADD' if is_today else '#D3D1C7'
            elw = 1.5 if is_today else 0.5
            if is_today: bg = '#E6F1FB'

            shapes.append(dict(
                type='rect', xref=xr, yref=yr,
                x0=x0+0.03, x1=x1-0.03, y0=yb+0.03, y1=yt-0.03,
                fillcolor=bg, line=dict(color=ec, width=elw), layer='below'
            ))

            # day number
            num_color = '#A32D2D' if is_weekend else '#444441'
            if not in_range: num_color = '#B4B2A9'
            if is_today:     num_color = '#185FA5'
            annotations.append(dict(
                xref=xr, yref=yr, x=x0+0.5, y=yt - CELL_PADDING - DAY_NUM_H / 2,
                text=f'<b>{day}</b>' if is_today else str(day),
                font=dict(color=num_color, size=10), showarrow=False
            ))

            # ── event pills — stacked top-down with consistent spacing ──
            visible_evts = day_evts[:MAX_PILLS]
            pill_top = yt - CELL_PADDING - DAY_NUM_H  # start just below day number

            for e_idx, evt in enumerate(visible_evts):
                label = evt[0]; time = evt[1]; cat = evt[2]
                desc  = evt[3] if len(evt) == 4 else None
                color = CATEGORY_COLORS.get(cat, '#888')

                p_top = pill_top - e_idx * (PILL_HEIGHT + PILL_GAP)
                p_bot = p_top - PILL_HEIGHT
                p_mid = (p_top + p_bot) / 2

                shapes.append(dict(
                    type='rect', xref=xr, yref=yr,
                    x0=x0+0.06, x1=x1-0.06, y0=p_bot, y1=p_top,
                    fillcolor=color, line_width=0, layer='above'
                ))

                pill_text = f'{time} {label}' if time else label
                hover_lines = [
                    f'<b>{label}</b>',
                    f'📅 {ds}',
                    f'🕐 {time}' if time else '📌 All day',
                    f'🏷 {cat.capitalize()}',
                ]
                if desc: hover_lines.append(f'📝 {desc}')

                trace = go.Scatter(
                    x=[x0 + 0.5], y=[p_mid],
                    mode='text',
                    text=[f'<span style="color:white;font-size:6.5px;font-weight:bold">{pill_text}</span>'],
                    textposition='middle center',
                    hovertemplate='<br>'.join(hover_lines) + '<extra></extra>',
                    hoverlabel=dict(
                        bgcolor='white', bordercolor=color,
                        font=dict(size=12, color='#2C2C2A')
                    ),
                    showlegend=False
                )
                traces_with_axes.append((trace, row_num, col_num))

            # overflow indicator
            if len(day_evts) > MAX_PILLS:
                overflow_y = pill_top - MAX_PILLS * (PILL_HEIGHT + PILL_GAP) - 0.05
                annotations.append(dict(
                    xref=xr, yref=yr, x=x0+0.5, y=overflow_y,
                    text=f'+{len(day_evts) - MAX_PILLS} more',
                    font=dict(color='#888780', size=7), showarrow=False
                ))

    return shapes, annotations, traces_with_axes, TOTAL_H

print('✅ Builder ready.')

✅ Builder ready.


## 📅 Render

In [4]:
start  = pd.to_datetime(START_DATE).date()
end    = pd.to_datetime(END_DATE).date()
today  = date.today()
months = get_months_in_range(start, end)
n      = len(months)
cols   = min(MONTHS_PER_ROW, n)
rows   = (n + cols - 1) // cols

fig = make_subplots(rows=rows, cols=cols,
                    horizontal_spacing=0.04,
                    vertical_spacing=0.06)

all_shapes, all_annotations = [], []

for i, (year, month) in enumerate(months):
    axis_idx = i + 1
    r = (i // cols) + 1
    c = (i %  cols) + 1

    shapes, annotations, traces_with_axes, total_h = build_month(
        year, month, start, end, EVENTS, today, axis_idx, cols
    )
    all_shapes      += shapes
    all_annotations += annotations

    for trace, tr, tc in traces_with_axes:
        fig.add_trace(trace, row=tr, col=tc)

    suf = axis_ref(axis_idx)
    fig.update_layout(**{
        f'xaxis{suf}': dict(range=[0,7], showgrid=False, zeroline=False,
                            showticklabels=False, fixedrange=True),
        f'yaxis{suf}': dict(range=[0,total_h], showgrid=False, zeroline=False,
                            showticklabels=False, fixedrange=True),
    })

for j in range(n, rows * cols):
    suf = axis_ref(j + 1)
    fig.update_layout(**{
        f'xaxis{suf}': dict(visible=False),
        f'yaxis{suf}': dict(visible=False),
    })

for cat, color in CATEGORY_COLORS.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=12, color=color, symbol='square'),
        name=cat.capitalize(), showlegend=True
    ))

cell_h = compute_cell_height()
fig.update_layout(
    title=dict(
        text=f'<b>{start.strftime("%B %d, %Y")}  →  {end.strftime("%B %d, %Y")}</b>',
        x=0.5, font=dict(size=16, color='#2C2C2A')
    ),
    shapes=all_shapes,
    annotations=all_annotations,
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=rows * int((6 * cell_h + 1.2) * 60) + 100,
    margin=dict(l=20, r=20, t=60, b=60),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.04,
                font=dict(size=12)),
    hovermode='closest',
)

fig.show()
fig.write_html('calendar_output.html')
print(f'✅ Done  |  {n} month(s)  |  {sum(len(v) for v in EVENTS.values())} events')

✅ Done  |  2 month(s)  |  15 events


In [5]:
start  = pd.to_datetime(START_DATE).date()
end    = pd.to_datetime(END_DATE).date()
today  = date.today()
months = get_months_in_range(start, end)

import calendar as cal_mod

DAY_NAMES_SHORT = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
DAY_NAMES_FULL  = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
MONTH_NAMES     = ['January','February','March','April','May','June',
                   'July','August','September','October','November','December']

# ── legend ────────────────────────────────────────────────────
legend_html = ''.join(
    f'<div class="legend-item">'
    f'<div class="legend-dot" style="background:{c}"></div>'
    f'<span>{k.capitalize()}</span></div>'
    for k, c in CATEGORY_COLORS.items()
)

# ── months ────────────────────────────────────────────────────
months_html = ''
card_id = 0  # unique id for each checkbox

for year, month in months:
    days_in_month = cal_mod.monthrange(year, month)[1]
    month_name    = MONTH_NAMES[month - 1]
    tstr          = today.strftime('%Y-%m-%d')

    # ── mini calendar ──────────────────────────────────────────
    first_dow  = date(year, month, 1).weekday()  # Mon=0
    mini_cells = '<div class="mini-day out"></div>' * first_dow

    for d in range(1, days_in_month + 1):
        ds         = date(year, month, d).strftime('%Y-%m-%d')
        dow        = date(year, month, d).weekday()
        in_range   = START_DATE <= ds <= END_DATE
        is_today   = ds == tstr
        has_events = ds in EVENTS
        is_weekend = dow >= 5
        cls = 'mini-day'
        if not in_range:             cls += ' out'
        if is_weekend and in_range:  cls += ' wknd'
        if is_today:                 cls += ' today'
        if has_events:               cls += ' has-events'
        mini_cells += f'<div class="{cls}"><span class="dnw">{d}</span></div>'

    mini_headers = ''.join(
        f'<div class="mini-dayname{" wknd" if i>=5 else ""}">{"MTWTFSS"[i]}</div>'
        for i in range(7)
    )

    # ── agenda rows ────────────────────────────────────────────
    agenda_rows = ''
    for d in range(1, days_in_month + 1):
        ds       = date(year, month, d).strftime('%Y-%m-%d')
        if ds < START_DATE or ds > END_DATE: continue
        day_evts = EVENTS.get(ds, [])
        if not day_evts: continue

        dow      = date(year, month, d).weekday()
        dow_sun  = (dow + 1) % 7
        is_today = ds == tstr
        day_name = DAY_NAMES_SHORT[dow_sun]
        full_day = DAY_NAMES_FULL[dow_sun]

        cards_html = ''
        for evt in day_evts:
            label = evt[0]
            time  = evt[1]
            cat   = evt[2]
            desc  = evt[3] if len(evt) > 3 else ''
            color = CATEGORY_COLORS.get(cat, '#888')
            time_label = f'🕐 {time}' if time else '📌 All day'
            time_val   = f'🕐 {time}' if time else '📌 All day'

            detail_lines = (
                f'📅 {full_day}, {MONTH_NAMES[month-1]} {d}, {year}<br>'
                f'{time_val}<br>'
                f'🏷 {cat.capitalize()}'
            )
            if desc:
                detail_lines += f'<br>📝 {desc}'

            cards_html += f'''<label>
              <input type="checkbox" id="c{card_id}">
              <div class="event-card" style="border-left-color:{color}">
                <div class="event-card-content">
                  <div class="event-card-label">{label}</div>
                  <div class="event-card-time">{time_label}</div>
                </div>
                <div class="chevron">›</div>
              </div>
              <div class="detail">{detail_lines}</div>
            </label>'''
            card_id += 1

        today_cls = ' is-today' if is_today else ''
        agenda_rows += f'''<div class="agenda-day{today_cls}">
          <div class="agenda-date-col">
            <div class="agenda-day-num">{d}</div>
            <div class="agenda-day-name">{day_name}</div>
          </div>
          <div class="agenda-events">{cards_html}</div>
        </div>'''

    months_html += f'''<div class="month-section">
      <div class="month-label">{month_name} {year}</div>
      <div class="mini-cal"><div class="mini-grid">{mini_headers}{mini_cells}</div></div>
      <div class="agenda">{agenda_rows if agenda_rows else '<div class="empty-state">No events this month</div>'}</div>
    </div>'''

title_str = f'{start.strftime("%b %d")} → {end.strftime("%b %d, %Y")}'

html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
<title>Calendar</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; -webkit-tap-highlight-color: transparent; }}
  body {{ font-family: -apple-system, BlinkMacSystemFont, sans-serif;
          background: #F2F2F7; padding-bottom: 40px; }}

  input[type=checkbox] {{ display: none; }}

  .top-bar {{ background: #2C2C2A; color: white; padding: 18px 16px 14px;
              text-align: center; position: sticky; top: 0; z-index: 10; }}
  .top-bar h1 {{ font-size: 17px; font-weight: 700; }}
  .top-bar p  {{ font-size: 13px; opacity: 0.7; margin-top: 3px; }}

  .legend {{ display: flex; flex-wrap: wrap; justify-content: center;
             gap: 8px 14px; padding: 12px 16px; background: white;
             border-bottom: 1px solid #E5E5EA; }}
  .legend-item {{ display: flex; align-items: center; gap: 6px;
                  font-size: 13px; color: #3A3A3C; }}
  .legend-dot  {{ width: 10px; height: 10px; border-radius: 50%; flex-shrink: 0; }}

  .month-section {{ margin-top: 20px; }}
  .month-label {{ font-size: 13px; font-weight: 700; color: #8E8E93;
                  text-transform: uppercase; letter-spacing: 0.5px;
                  padding: 0 16px 6px; }}

  .mini-cal  {{ background: white; margin-bottom: 2px; }}
  .mini-grid {{ display: grid; grid-template-columns: repeat(7, 1fr); }}
  .mini-dayname {{ text-align: center; font-size: 11px; font-weight: 600;
                   color: #8E8E93; padding: 8px 0 4px; }}
  .mini-dayname.wknd {{ color: #A32D2D; }}
  .mini-day {{ text-align: center; padding: 4px 2px 6px;
               font-size: 13px; color: #3A3A3C; }}
  .mini-day.out  {{ color: #C7C7CC; }}
  .mini-day.wknd {{ color: #A32D2D; }}
  .mini-day.today .dnw {{
    background: #378ADD; color: white; border-radius: 50%;
    width: 26px; height: 26px; display: inline-flex;
    align-items: center; justify-content: center; font-weight: 700; }}
  .mini-day.has-events::after {{
    content: ""; display: block; width: 5px; height: 5px;
    border-radius: 50%; background: #378ADD; margin: 2px auto 0; }}
  .mini-day.today.has-events::after {{ background: white; }}

  .agenda {{ padding: 0 16px; margin-top: 12px; }}
  .agenda-day {{ display: flex; gap: 12px; margin-bottom: 16px; }}
  .agenda-date-col {{ width: 42px; flex-shrink: 0; text-align: center; padding-top: 2px; }}
  .agenda-day-num  {{ font-size: 22px; font-weight: 300; color: #1C1C1E; line-height: 1; }}
  .agenda-day-name {{ font-size: 11px; color: #8E8E93; text-transform: uppercase;
                      letter-spacing: 0.3px; margin-top: 2px; }}
  .agenda-day.is-today .agenda-date-col {{
    display: flex; flex-direction: column; align-items: center; }}
  .agenda-day.is-today .agenda-day-num {{
    color: white; background: #378ADD; border-radius: 50%;
    width: 36px; height: 36px; display: inline-flex;
    align-items: center; justify-content: center;
    font-size: 18px; font-weight: 600; }}
  .agenda-events {{ flex: 1; display: flex; flex-direction: column; gap: 6px; }}

  label {{ display: block; }}

  .event-card {{
    background: white; border-radius: 12px; padding: 12px 14px;
    display: flex; align-items: center; gap: 12px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.08);
    border-left: 4px solid transparent; cursor: pointer;
  }}
  .event-card:active {{ opacity: 0.7; }}
  .event-card-content {{ flex: 1; min-width: 0; }}
  .event-card-label {{ font-size: 15px; font-weight: 600; color: #1C1C1E; }}
  .event-card-time  {{ font-size: 13px; color: #8E8E93; margin-top: 2px; }}
  .chevron {{ color: #C7C7CC; font-size: 20px; flex-shrink: 0;
              transition: transform 0.2s; }}

  .detail {{
    display: none;
    background: #F9F9F9; border-radius: 10px;
    padding: 12px 14px; margin-top: 4px;
    border: 1px solid #E5E5EA;
    font-size: 14px; color: #3A3A3C; line-height: 1.8;
  }}

  input:checked + .event-card {{ border-left-width: 6px; }}
  input:checked + .event-card .chevron {{ transform: rotate(90deg); color: #555; }}
  input:checked + .event-card + .detail {{ display: block; }}

  .empty-state {{ text-align: center; padding: 20px; color: #8E8E93; font-size: 14px; }}
</style>
</head>
<body>

<div class="top-bar">
  <h1>📅 My Calendar</h1>
  <p>{title_str}</p>
</div>

<div class="legend">{legend_html}</div>
<div>{months_html}</div>

</body>
</html>'''

with open('calendar_mobile.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('✅ calendar_mobile.html saved — tap any event to expand details!')

✅ calendar_mobile.html saved — tap any event to expand details!
